In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression  # Example classifier

# Generate synthetic data (two views)
np.random.seed(0)
n_samples = 100
X1 = np.random.randn(n_samples, 2) + np.array([2, 2])  # View 1
X2 = X1 + np.random.randn(n_samples, 2) * 0.5  # View 2 (slightly noisy version of View 1)
y = (X1[:, 0] > 2).astype(int)  # True labels (based on View 1)

# Split data into labeled and unlabeled sets
n_labeled = 20  # Start with a small labeled set
labeled_indices = np.random.choice(n_samples, n_labeled, replace=False)
unlabeled_indices = np.array(list(set(range(n_samples)) - set(labeled_indices)))

X1_labeled = X1[labeled_indices]
X2_labeled = X2[labeled_indices]
y_labeled = y[labeled_indices]

X1_unlabeled = X1[unlabeled_indices]
X2_unlabeled = X2[unlabeled_indices]

# Initialize classifiers (one for each view)
clf1 = LogisticRegression()
clf2 = LogisticRegression()

# Co-training loop
n_iterations = 10  # Number of co-training iterations
for _ in range(n_iterations):
    # Train classifiers on labeled data
    clf1.fit(X1_labeled, y_labeled)
    clf2.fit(X2_labeled, y_labeled)

    # Predict on unlabeled data
    y1_unlabeled_pred = clf1.predict(X1_unlabeled)
    y2_unlabeled_pred = clf2.predict(X2_unlabeled)

    # Find high-confidence predictions (where both classifiers agree)
    agree_indices = np.where(y1_unlabeled_pred == y2_unlabeled_pred)[0]

    if len(agree_indices) > 0: #Check if there are any agreeing indices
        # Select a subset of agreed upon examples to add to the labeled set
        n_to_add = min(10, len(agree_indices))  # Add up to 10 examples
        add_indices = np.random.choice(agree_indices, n_to_add, replace=False)

        X1_added = X1_unlabeled[add_indices]
        X2_added = X2_unlabeled[add_indices]
        y_added = y1_unlabeled_pred[add_indices]  # Use the agreed-upon label

        # Update labeled data
        X1_labeled = np.concatenate([X1_labeled, X1_added])
        X2_labeled = np.concatenate([X2_labeled, X2_added])
        y_labeled = np.concatenate([y_labeled, y_added])

        # Remove added examples from unlabeled data
        remaining_indices = np.array(list(set(range(len(X1_unlabeled))) - set(add_indices)))
        X1_unlabeled = X1_unlabeled[remaining_indices]
        X2_unlabeled = X2_unlabeled[remaining_indices]

# Plot the results
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.scatter(X1[:, 0], X1[:, 1], c=y, label='True Labels')
plt.title('Original Data (View 1)')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(X1_labeled[:, 0], X1_labeled[:, 1], c=y_labeled, label='Co-trained Labels')
plt.title('Co-training Results (View 1)')
plt.legend()

plt.show()

# Optional: Plot View 2 as well (similar to View 1 plots).
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.scatter(X2[:, 0], X2[:, 1], c=y, label='True Labels')
plt.title('Original Data (View 2)')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(X2_labeled[:, 0], X2_labeled[:, 1], c=y_labeled, label='Co-trained Labels')
plt.title('Co-training Results (View 2)')
plt.legend()

plt.show()
